# PathSegmentor: run your first prediction

First complete the environment setup in [README.md](README.md). Start Jupyter from the repository root and select the **PathSegmentor** kernel. Place the downloaded weights at `checkpoints/model_state_dict.pt`, then choose **Run All**. A CUDA-capable NVIDIA GPU is required. The first model load downloads the CLIP tokenizer if it is not cached.

The next cell contains all settings. Keep the bundled image and prompt for your first run. No training dataset or ground-truth mask is needed. To use your own image, change `image_path` and `text_prompt`; prompts must be a nonempty list of strings. The demo pads and resizes images to 1024 x 1024.

The first visible GPU is used automatically. To choose another GPU, set `CUDA_VISIBLE_DEVICES` before starting Jupyter (see README). After changing the checkpoint, restart the kernel and run all cells again.

In [ ]:
from pathlib import Path

# Settings: keep these defaults for the bundled example.
pretrained_pth = 'checkpoints/model_state_dict.pt'
image_path = 'examples/1_irregular/2/test181_pathology_colon.png'
text_prompt = ['tissue-level gland in colon pathology']

if not Path('configs/pathsegmentor_inference.yaml').is_file():
    raise FileNotFoundError('Start Jupyter from the PathSegmentor repository root; see README.md.')
if not Path(pretrained_pth).is_file():
    raise FileNotFoundError(f'Download the checkpoint linked in README.md and place it at {pretrained_pth}, or update pretrained_pth.')
if not Path(image_path).is_file():
    raise FileNotFoundError(f'Image not found: {image_path}. Update image_path in this cell.')
if not isinstance(text_prompt, list) or not text_prompt or not all(isinstance(t, str) and t.strip() for t in text_prompt):
    raise ValueError('text_prompt must be a nonempty list of nonempty strings.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select the PathSegmentor kernel and check the NVIDIA driver and GPU visibility; see README.md.')
print(f'Using GPU: {torch.cuda.get_device_name(0)}')

from PIL import Image
from modeling.BaseModel import BaseModel
from modeling import build_model
from utilities.distributed import init_distributed
from utilities.arguments import load_opt_from_config_files
from utilities.constants import PATHSEG_CLASSES

from inference_utils.inference import interactive_infer_image

opt = load_opt_from_config_files(["configs/pathsegmentor_inference.yaml"])
opt = init_distributed(opt)

# Load model from pretrained weights
model = BaseModel(opt, build_model(opt)).from_pretrained(pretrained_pth).eval().cuda()
with torch.no_grad():
    model.model.sem_seg_head.predictor.lang_encoder.get_text_embeddings(PATHSEG_CLASSES + ["background"], is_eval=True)
print('Model ready. Continue with the remaining cells.')

### Utility Functions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.morphology import binary_erosion

def plot_segmentation_masks(original_image, segmentation_masks, texts, 
                            colors=None, alpha=0.6, boundary_thickness=3, gt_masks=None):
    ''' 
    Plot the original image, predicted mask, and GT mask overlay separately (three panels per row) using the same color.
    
    Args:
        original_image (ndarray): The original RGB image.
        segmentation_masks (list): The predicted masks.
        texts (list): Text labels for each mask.
        colors (list): Colors for the masks; defaults to red.
        alpha (float): Mask transparency.
        boundary_thickness (int): Boundary line thickness.
        gt_masks (list, optional): GT masks corresponding one-to-one with segmentation_masks.
    '''
    original_image = original_image[:, :, :3].astype(np.uint8)

    if colors is None:
        colors = [[255, 0, 0]] * len(segmentation_masks)

    num_cols = 3 if gt_masks is not None else 2
    fig, ax = plt.subplots(len(segmentation_masks), num_cols, figsize=(4 * num_cols, 4 * len(segmentation_masks)))

    if len(segmentation_masks) == 1:
        ax = np.expand_dims(ax, axis=0)  # Ensure compatibility with two-dimensional indexing.

    for i, (pred_mask, label) in enumerate(zip(segmentation_masks, texts)):
        mask_color = np.array(colors[i]).reshape(1, 1, 3)

        # ----------------- Original image -----------------
        ax[i][0].imshow(original_image)
        ax[i][0].set_title('Original Image')
        ax[i][0].axis('off')

        # ----------------- Predicted mask -----------------
        pred_binary = pred_mask > 0.5
        pred_overlay = original_image.copy()

        # Alpha blending
        pred_overlay[pred_binary] = (
            (1 - alpha) * pred_overlay[pred_binary] + alpha * mask_color
        ).astype(np.uint8)

        # Predicted mask boundary.
        current_mask = pred_binary.copy()
        pred_boundary = np.zeros_like(pred_binary)
        for _ in range(boundary_thickness):
            eroded = binary_erosion(current_mask)
            boundary = current_mask & (~eroded)
            pred_boundary |= boundary
            current_mask = eroded

        pred_overlay[pred_boundary] = mask_color
        ax[i][1].imshow(pred_overlay)
        ax[i][1].set_title('Pred Mask')
        ax[i][1].axis('off')

        # ----------------- GT mask -----------------
        if gt_masks is not None:
            gt_mask = gt_masks[i] > 0.5
            gt_overlay = original_image.copy()

            # Alpha blending
            gt_overlay[gt_mask] = (
                (1 - alpha) * gt_overlay[gt_mask] + alpha * mask_color
            ).astype(np.uint8)

            # GT mask boundary.
            current_gt = gt_mask.copy()
            gt_boundary = np.zeros_like(gt_mask)
            for _ in range(boundary_thickness):
                eroded_gt = binary_erosion(current_gt)
                boundary = current_gt & (~eroded_gt)
                gt_boundary |= boundary
                current_gt = eroded_gt

            gt_overlay[gt_boundary] = mask_color
            ax[i][2].imshow(gt_overlay)
            ax[i][2].set_title('GT Mask')
            ax[i][2].axis('off')

    plt.tight_layout()
    plt.show()

## Run inference

Run the following cells to display the input image and predicted mask overlay for each prompt. To try another image after loading the model, update `image_path` and `text_prompt` here and rerun the final cell; no model reload is needed. Ground-truth masks in `examples/` are optional reference annotations and are not used by this demo.

In [ ]:
from inference_utils.processing_utils import read_rgb


def inference_rgb(file_path, text_prompts):
    image = read_rgb(file_path)
    
    pred_mask = interactive_infer_image(model, Image.fromarray(image), text_prompts)
    
    return image, pred_mask

In [ ]:

image, pred_mask = inference_rgb(image_path, text_prompt)

plot_segmentation_masks(image, pred_mask, text_prompt)